# 각 파일에서 6가지 점수 데이터 불러오기

In [11]:
# -*- coding: utf-8 -*-
"""
EWS 종합 위험 스코어용 변수 통합 스크립트 (수정본)
==========================================

[수정 내역]
 1. minmax_by_year 중복곱셈(*100*100 -> 0~10000 스케일) 버그 수정
    -> minmax() 내부에서 이미 *100을 하므로, minmax_by_year에서는 추가로 곱하지 않음
 2. 최종 출력 시 모든 점수 컬럼을 소수점 둘째자리까지 반올림
 3. (참고) 회사명 결측 / 기준연도-회계년도 불일치는 코드 버그가 아니라
    데이터셋 간 표본범위 차이 및 재무자료 입수 시차(reporting lag) 때문입니다.
    -> 자세한 설명은 채팅 답변 참고
"""

import pandas as pd
import numpy as np
import os

# ------------------------------------------------------------------
# 0. 설정
# ------------------------------------------------------------------

OUT_PATH = os.path.join("22번 스코어 산출\통합_스코어_데이터.csv")

# True로 바꾸면 기업단위 변수(기업베타/부실확률/diff)도
# '연도별 횡단면 정규화' 대신 '10개 연도 전체 통합 min-max'를 사용합니다.
GLOBAL_MINMAX = False


def read_csv_safe(filename, **kwargs):
    """사업자등록번호를 문자열(10자리 zfill)로 안전하게 읽는 CSV 로더"""
    path = os.path.join(filename)
    df = pd.read_csv(path, dtype={"사업자등록번호": str}, **kwargs)
    if "사업자등록번호" in df.columns:
        df["사업자등록번호"] = df["사업자등록번호"].str.zfill(10)
    return df


def minmax(s: pd.Series) -> pd.Series:
    """NaN을 무시하는 0~100 스케일 min-max. max==min이면 50으로 처리(분산 0)."""
    mn, mx = s.min(skipna=True), s.max(skipna=True)
    if pd.isna(mn) or pd.isna(mx) or mx == mn:
        return pd.Series(np.where(s.notna(), 50.0, np.nan), index=s.index)
    return (s - mn) / (mx - mn) * 100


def minmax_by_year(df, value_col, year_col="연도"):
    """연도별(횡단면) 0~100 스케일 min-max.
    주의: minmax() 내부에서 이미 *100을 하므로 여기서 추가로 곱하면 안 됨 (중복곱셈 버그 수정됨)
    """
    return df.groupby(year_col)[value_col].transform(minmax)


# ------------------------------------------------------------------
# 1. 베이스 테이블: lifecycle_scored_yearly_minmax.csv
#    (사업자등록번호, 기준연도) 단위가 고유(unique) -> '기준연도'를 '연도'로 사용
# ------------------------------------------------------------------
lifecycle = read_csv_safe("20번. 기업 생애주기\lifecycle_scored_yearly_minmax.csv")

base = lifecycle.rename(columns={"기준연도": "연도"})[
    ["사업자등록번호", "연도", "회계년도", "생애주기_최종", "생애주기_점수", "부실라벨_ICR3년"]
].copy()

# ------------------------------------------------------------------
# 2. 마이클 포터 5F (산업 단위, 도소매업) -> 연도 기준 broadcast
# ------------------------------------------------------------------
porter = read_csv_safe("19번 마이클 포터\마이클 포터 5F_도소매업.csv")
porter = porter[["연도", "최종점수"]].copy()
porter["porter_5F_minmax"] = minmax(porter["최종점수"])   # 10개 연도 시계열 min-max
porter = porter[["연도", "porter_5F_minmax"]]

# ------------------------------------------------------------------
# 3. 산업충격민감도_OLS (산업 단위, 도소매업) -> 연도 기준 broadcast
# ------------------------------------------------------------------
ind_ols = read_csv_safe("18번 산업별 충격민감도\산업충격민감도_OLS.csv")
ind_ols = ind_ols.rename(columns={"테스트_연도": "연도"})[["연도", "beta_i"]].copy()
ind_ols["산업베타_minmax"] = minmax(ind_ols["beta_i"])    # 10개 연도 시계열 min-max
ind_ols = ind_ols[["연도", "산업베타_minmax"]]

# ------------------------------------------------------------------
# 4. 충격민감도_OLS (기업 단위) -> (사업자등록번호, 연도) 기준
# ------------------------------------------------------------------
firm_ols = read_csv_safe("17번 기업별 충격민감도\충격민감도_OLS.csv")
firm_ols = firm_ols.rename(columns={"테스트_연도": "연도"})[
    ["사업자등록번호", "회사명", "연도", "beta_i"]
].copy()

if GLOBAL_MINMAX:
    firm_ols["기업베타_minmax"] = minmax(firm_ols["beta_i"])
else:
    firm_ols["기업베타_minmax"] = minmax_by_year(firm_ols, "beta_i")

firm_ols_for_merge = firm_ols[["사업자등록번호", "연도", "기업베타_minmax"]]
firm_name_map = firm_ols[["사업자등록번호", "회사명"]].drop_duplicates(subset=["사업자등록번호"])

# ------------------------------------------------------------------
# 5. 부실확률 차이값 (기업 단위, wide -> long 변환)
#    연도 범위: 2015~2024 (diff_2014_2015 ~ diff_2023_2024 이용)
# ------------------------------------------------------------------
prob = read_csv_safe(r"21번. 기업 PD 변화율\2015-2024_기업_부실확률_차이값.csv")

years = range(2015, 2025)
long_rows = []
for y in years:
    prob_col = f"prob_{y}"
    diff_col = f"diff_{y-1}_{y}"
    tmp = prob[["사업자등록번호", "회사명", prob_col, diff_col]].copy()
    tmp.columns = ["사업자등록번호", "회사명", "prob", "diff"]
    tmp["연도"] = y
    long_rows.append(tmp)

prob_long = pd.concat(long_rows, ignore_index=True)

if GLOBAL_MINMAX:
    prob_long["부실확률_minmax"] = minmax(prob_long["prob"])
    prob_long["부실확률변화_minmax"] = minmax(prob_long["diff"])
else:
    prob_long["부실확률_minmax"] = minmax_by_year(prob_long, "prob")
    prob_long["부실확률변화_minmax"] = minmax_by_year(prob_long, "diff")

prob_for_merge = prob_long[
    ["사업자등록번호", "회사명", "연도", "부실확률_minmax", "부실확률변화_minmax"]
]

# ------------------------------------------------------------------
# 6. 전체 병합
# ------------------------------------------------------------------
df = base.copy()

# 6-1. 산업 단위 변수 (연도 기준 broadcast)
df = df.merge(porter, on="연도", how="left")
df = df.merge(ind_ols, on="연도", how="left")

# 6-2. 기업 단위 변수
df = df.merge(firm_ols_for_merge, on=["사업자등록번호", "연도"], how="left")
df = df.merge(prob_for_merge, on=["사업자등록번호", "연도"], how="left", suffixes=("", "_prob"))

# 회사명 채우기: 부실확률 파일의 회사명을 우선 사용하고, 없는 경우만 충격민감도_OLS 매핑으로 보강
# (※ if/else로 분기하면, df에 '회사명'이 아직 없을 때 merge 결과 컬럼명이 '회사명_prob'가 아닌
#    '회사명'으로 들어가 else 분기가 타면서 기존 값을 덮어써버리는 버그가 있어 -> 항상 combine_first로 통일)
df["회사명"] = df["회사명"].combine_first(
    df["사업자등록번호"].map(firm_name_map.set_index("사업자등록번호")["회사명"])
)

# ------------------------------------------------------------------
# 6-3. 최종 출력용 컬럼명으로 변경
# ------------------------------------------------------------------
RENAME_MAP = {
    "porter_5F_minmax": "Porter5F",
    "생애주기_점수": "생애주기점수",
    "산업베타_minmax": "산업충격민감도",
    "기업베타_minmax": "기업충격민감도",
    "부실확률_minmax": "부실확률",
    "부실확률변화_minmax": "부실확률변화",
}
df = df.rename(columns=RENAME_MAP)

# ------------------------------------------------------------------
# 7. 최종 컬럼 정리, 소수점 둘째자리 반올림 및 저장
# ------------------------------------------------------------------
final_cols = [
    "사업자등록번호", "회사명", "연도", 
    "생애주기_최종", "부실라벨_ICR3년",
    "Porter5F",          
    "생애주기점수",        
    "산업충격민감도",      
    "기업충격민감도",     
    "부실확률",            
    "부실확률변화",       
]

df_final = df[final_cols].sort_values(["사업자등록번호", "연도"]).reset_index(drop=True)

# 점수 컬럼만 소수점 둘째자리로 반올림
score_cols = [
    "생애주기점수", "Porter5F", "산업충격민감도",
    "기업충격민감도", "부실확률", "부실확률변화",
]
df_final[score_cols] = df_final[score_cols].round(2)

# ------------------------------------------------------------------
# 7-1. '부실확률'이 없는 행 삭제
# ------------------------------------------------------------------
before_n = len(df_final)
df_final = df_final.dropna(subset=["부실확률"]).reset_index(drop=True)
after_n = len(df_final)
print(f"\n'부실확률' 결측 행 삭제: {before_n} -> {after_n} ({before_n - after_n}행 제거)")

df_final.to_csv(OUT_PATH, index=False, encoding="utf-8-sig")

print(f"완료: {OUT_PATH}")
print(f"행/열: {df_final.shape}")
print(df_final.head(10))
print("\n[결측치 비율]")
print(df_final.isna().mean().round(3))
print("\n[점수 컬럼 범위 확인 - 정상적으로 0~100 사이여야 함]")
print(df_final[score_cols].describe().loc[['min','max']])

<>:58: SyntaxWarning: invalid escape sequence '\l'
<>:58: SyntaxWarning: invalid escape sequence '\l'
C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_15516\2687088149.py:58: SyntaxWarning: invalid escape sequence '\l'
  lifecycle = read_csv_safe("20번. 기업 생애주기\lifecycle_scored_yearly_minmax.csv")



'부실확률' 결측 행 삭제: 48057 -> 32801 (15256행 제거)
완료: 22번 스코어 산출\통합_스코어_데이터.csv
행/열: (32801, 11)
      사업자등록번호          회사명    연도 생애주기_최종  부실라벨_ICR3년  Porter5F  생애주기점수  \
0  0008110041  엘브이엠씨홀딩스(주)  2015     도입기           0    100.00    75.7   
1  0008110041  엘브이엠씨홀딩스(주)  2016     도입기           0     56.26    80.6   
2  0008110041  엘브이엠씨홀딩스(주)  2017     성장기           0     34.84     1.1   
3  0008110041  엘브이엠씨홀딩스(주)  2018     성장기           0     25.54     4.0   
4  0008110041  엘브이엠씨홀딩스(주)  2019     성장기           0     56.69     0.2   
5  0008110041  엘브이엠씨홀딩스(주)  2020     성장기           1      0.00     0.0   
6  1018106586      서원물산(주)  2015     도입기           1    100.00    75.7   
7  1018116269   현대코퍼레이션(주)  2015     성숙기           0    100.00     3.2   
8  1018116269   현대코퍼레이션(주)  2016     성장기           0     56.26     0.3   
9  1018116269   현대코퍼레이션(주)  2017     조정기           0     34.84    21.8   

   산업충격민감도  기업충격민감도   부실확률  부실확률변화  
0     0.00    44.36   0.19   49.99  
1    42.55    48.52 

# 가중치 곱해 종합점수 산출

In [12]:
import pandas as pd

score_df = pd.read_csv("22번 스코어 산출/통합_스코어_데이터.csv")

other_cols = ["Porter5F", "생애주기점수", "산업충격민감도", "기업충격민감도", "부실확률변화"]

# 부실확률 가중치 후보: 0.4, 0.5, 0.6, 0.7, 0.8
prob_weights = [0.4, 0.5, 0.6, 0.7, 0.8]

for w in prob_weights:
    other_w = (1 - w) / len(other_cols)  # 나머지 5개 변수에 동일 분배

    col_name = f"종합점수_{w}"
    score_df[col_name] = (
        score_df["부실확률"] * w
        + score_df[other_cols].sum(axis=1) * other_w
    ).round(2)

score_df.to_csv("22번 스코어 산출/통합_스코어_데이터.csv", index=False, encoding="utf-8-sig")

# 신용등급 존재하는 데이터 merge

In [14]:
import pandas as pd

credit_df = pd.read_excel("..\데이터수집\신용등급\상장사 신용등급.xlsx")


# ============================================================
# 2. 신용등급 파일 전처리
# ============================================================
# 사업자등록번호: "134-81-71976 " -> 1348171976
#   (하이픈/공백 제거 후 int 변환 -> 통합_스코어_데이터의 사업자등록번호 형식과 통일)
credit_df["사업자등록번호"] = (
    credit_df["사업자등록번호"]
    .astype(str).str.strip()
    .str.replace("-", "", regex=False)
    .astype("int64")
)

# 회계년도: "2015/12" -> 2015 (통합_스코어_데이터의 '연도'와 동일한 형식으로 맞춤)
credit_df["연도"] = credit_df["회계년도"].astype(str).str.split("/").str[0].astype(int)

# 평가사구분 10(NICE신용평가) / 20(한국신용평가) / 30(한국기업평가)만 사용
credit_filtered = credit_df[credit_df["평가사구분"].isin([10])].copy()

# 동일 (사업자등록번호, 연도, 평가사명, 신용등급) 중복행 제거
credit_filtered = credit_filtered.drop_duplicates(
    subset=["사업자등록번호", "연도", "평가사명 및 등급", "신용등급"]
)

credit_for_merge = credit_filtered[
    ["사업자등록번호", "연도", "평가사명 및 등급", "신용등급", "평가사구분"]
]


# ============================================================
# 3. 병합 (사업자등록번호 + 연도 동일하고, 평가사구분 10/20/30이 존재하는 경우만 -> inner join)
# ============================================================
result_df = score_df.merge(credit_for_merge, on=["사업자등록번호", "연도"], how="inner")


# ============================================================
# 4. 최종 컬럼 정리
# ============================================================
final_cols = [
    "사업자등록번호", "회사명", "연도",
    "종합점수_0.4", "종합점수_0.5", "종합점수_0.6", "종합점수_0.7", "종합점수_0.8",
    "평가사명 및 등급", "신용등급", "평가사구분",
]
result_df = result_df[final_cols]

print(f"결과 shape: {result_df.shape}")
print(f"고유 (사업자등록번호, 연도) 수: {result_df[['사업자등록번호','연도']].drop_duplicates().shape[0]}")
print(result_df.head(10))

result_df.to_csv("22번 스코어 산출\통합_스코어_신용등급_병합.csv", index=False, encoding="utf-8-sig")

결과 shape: (420, 11)
고유 (사업자등록번호, 연도) 수: 227
      사업자등록번호         회사명    연도  종합점수_0.4  종합점수_0.5  종합점수_0.6  종합점수_0.7  \
0  1018116269  현대코퍼레이션(주)  2019     29.64     24.85     20.05     15.25   
1  1018116269  현대코퍼레이션(주)  2020     20.93     17.46     13.98     10.51   
2  1018116269  현대코퍼레이션(주)  2021     41.35     34.53     27.71     20.89   
3  1018116269  현대코퍼레이션(주)  2022     32.05     26.74     21.43     16.11   
4  1018116269  현대코퍼레이션(주)  2023     42.52     35.47     28.42     21.38   
5  1018116269  현대코퍼레이션(주)  2023     42.52     35.47     28.42     21.38   
6  1018116269  현대코퍼레이션(주)  2023     42.52     35.47     28.42     21.38   
7  1018116269  현대코퍼레이션(주)  2024     37.39     31.17     24.95     18.73   
8  1018116269  현대코퍼레이션(주)  2024     37.39     31.17     24.95     18.73   
9  1018116269  현대코퍼레이션(주)  2024     37.39     31.17     24.95     18.73   

   종합점수_0.8        평가사명 및 등급         신용등급  평가사구분  
0     10.45  NICE신용평가(AAA~D)    A-/STABLE   10.0  
1      7.04  NICE신용평가(AAA~D)